
### GROUP MEMBERS:
- Otieno Patience Atieno - 159442
- Omwoyo Leo Kinara - 169956
- Corban Chogo - 165558


In [ ]:
food_df = pd.read_csv('food.csv')
ratings_df = pd.read_csv('ratings.csv')
df_merged = pd.merge(ratings_df, food_df, on='Food_ID', how='inner')


Question 2 : Food Choices and Nutrition Dataset  
Expected Output Variable: Food_ID/Name  
This is what will be recommended to the user based on their query that could include cuisine category,vegetarian restriction and target ingredients

#3a)

Number of Rows:511

Number of Columns:7


In [ ]:
import pandas as pd

df=df_merged

rows, columns = df_merged.shape
print(f"Number of Rows: {rows}")
print(f"Number of Columns: {columns}")

Number of Rows: 511
Number of Columns: 7


#3b)Column Datatypes
User_ID:     float64  
Food_ID:      float64   
Rating:float64  
Name:     object  
C_Type:      object  
Veg_Non:     object  
Describe:    object  


In [ ]:
print("Data Types Breakdown:")
print(df.dtypes.value_counts())

print("\nDetailed Column Datatypes:")
print(df.dtypes)

Data Types Breakdown:
object     4
float64    3
Name: count, dtype: int64

Detailed Column Datatypes:
User_ID     float64
Food_ID     float64
Rating      float64
Name         object
C_Type       object
Veg_Non      object
Describe     object
dtype: object


#3c)Missing Values Per Column:    
**Food_ID:**   0   
**Name:**  0  
**C_Type:**     0   
**Veg_Non:**     0  
**Describe**:    0  

In [ ]:
total_missing = df.isnull().sum().sum()
missing_per_col = df.isnull().sum()

print("Missing Values Per Column:")
print(missing_per_col)
print(f"\nTotal Missing Values: {total_missing}")

if total_missing == 0:
  print("Verdict: The dataset is complete (contains zero missing values).")
else:
  print(
      f"Verdict: The dataset is NOT complete; it contains {total_missing}"
      " missing values."
  )


Missing Values Per Column:
User_ID     0
Food_ID     0
Rating      0
Name        0
C_Type      0
Veg_Non     0
Describe    0
dtype: int64

Total Missing Values: 0
Verdict: The dataset is complete (contains zero missing values).


#3d)

In [ ]:
# Question 3d:

first_15 = df_merged.head(15)
last_20 = df_merged.tail(20)

# Merge both slices vertically
df_sample = pd.concat([first_15, last_20], ignore_index=True)

print(f"Shape of df_sample: {df_sample.shape}")
df_sample

Shape of df_sample: (35, 7)


,User_ID,Food_ID,Rating,Name,C_Type,Veg_Non,Describe
0,1.0,88.0,4.0,peri peri chicken satay,Snack,non-veg,"boneless skinless chicken thigh (trimmed), sal..."
1,1.0,46.0,3.0,steam bunny chicken bao,Japanese,non-veg,"buns, all purpose white flour, dry yeast, suga..."
2,1.0,24.0,5.0,green lentil dessert fudge,Dessert,veg,"whole moong beans, cow ghee, raisins, whole mi..."
3,1.0,25.0,4.0,cashew nut cookies,Dessert,veg,"cashew paste, ghee, khaand (a sweetening agent..."
4,2.0,49.0,1.0,christmas tree pizza,Italian,veg,"pizza dough (2 boules), red pepper, red onion,..."
5,2.0,33.0,8.0,moong dal kiwi coconut soup,Indian,veg,"green gram (dhuli moong dal), kiwis, coconut c..."
6,2.0,106.0,9.0,chicken nimbu dhaniya shorba,Beverage,non-veg,"water, chicken (diced)), ginger garlic paste, ..."
7,2.0,71.0,8.0,carrot ginger soup,Healthy Food,veg,"Carrots, Olive Oil, Salt, Vegetable Stock, Gin..."
8,3.0,73.0,9.0,hot chocolate,Beverage,veg,"milk, chocolate, cocoa powder, powdered sugar,..."
9,3.0,110.0,10.0,chicken and mushroom lasagna,Italian,non-veg,"chicken, salt, crush black pepper, garlic clov..."


In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import os

def load_data(food_path='food.csv'):
    """Loads dataset and fills any blank descriptions with fallback error handling."""
    if not os.path.exists(food_path):
        raise FileNotFoundError(
            f"Could not find '{food_path}'. Please make sure you have uploaded "
            "the file to your Colab session storage using the folder icon on the left."
        )
    df = pd.read_csv(food_path)
    df['Describe'] = df['Describe'].fillna('')
    return df

def recommend_food(user_query, df, is_veg=None, cuisine=None, top_n=5):
    """
    Recommends top matching dishes based on user search query and hard filters.

    Parameters:
    - user_query (str): Target ingredients or search text (e.g. "chicken garlic").
    - is_veg (str/bool): 'veg' to restrict to vegetarian, 'non-veg' for non-veg, or None.
    - cuisine (str): Filter by specific C_Type (e.g. 'Italian', 'Indian') or None.
    - top_n (int): Number of recommendations to return.
    """
    filtered_df = df.copy()

    # 1. Apply Hard Pre-Filters
    if is_veg is True or is_veg == 'veg':
        filtered_df = filtered_df[filtered_df['Veg_Non'].str.lower() == 'veg']
    elif is_veg is False or is_veg == 'non-veg':
        filtered_df = filtered_df[filtered_df['Veg_Non'].str.lower() == 'non-veg']

    if cuisine:
        filtered_df = filtered_df[filtered_df['C_Type'].str.lower() == cuisine.lower()]

    if filtered_df.empty:
        return "No dishes match the selected dietary or cuisine filters."

    # 2. Vectorize Ingredient Text using TF-IDF
    tfidf = TfidfVectorizer(stop_words='english')
    # Fit vectorizer on filtered dish descriptions
    tfidf_matrix = tfidf.fit_transform(filtered_df['Describe'])

    # 3. Vectorize the User Query
    query_vector = tfidf.transform([user_query])

    # 4. Calculate Cosine Similarity Scores
    similarity_scores = cosine_similarity(query_vector, tfidf_matrix).flatten()

    # 5. Attach scores and rank top recommendations
    filtered_df = filtered_df.copy()
    filtered_df['Similarity_Score'] = similarity_scores

    # Sort by highest score first
    results = filtered_df.sort_values(by='Similarity_Score', ascending=False).head(top_n)

    return results[['Food_ID', 'Name', 'C_Type', 'Veg_Non', 'Describe', 'Similarity_Score']]

# --- Local Test Run ---
if __name__ == "__main__":
    try:
        # Load dataset
        food_df = load_data('food.csv')

        # Test Query: Search for spicy chicken dishes, enforcing non-veg filter
        query = "rice and beef"
        recommendations = recommend_food(user_query=query, df=food_df, is_veg='non-veg', top_n=3)

        print("--- Top Recommendations ---")
        print(recommendations)
    except FileNotFoundError as e:
        print(f"Error: {e}")

--- Top Recommendations ---
     Food_ID                     Name     C_Type  Veg_Non  \
321      322        Pho Tai rare beef  Vietnames  non-veg   
368      369            Beef Bibimbab     Korean  non-veg   
392      393  Braised Beef Short Ribs     Korean  non-veg   

                                              Describe  Similarity_Score  
321  Bean Sprouts, lemon, Thai basils, Rice noodles...          0.448861  
368  Beef,Cucumber,carrot, mushrooms,Spinich,Rice, ...          0.437656  
392                  Beef, Scallion, Raddish, Spanich           0.377710  


In [6]:
import re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.multiclass import OneVsRestClassifier

# 1. Seed lexicons covering both dataset vocabulary and local Kenyan staples
NUTRIENT_KEYWORDS = {
    'Protein': [
        'chicken', 'beef', 'pork', 'mutton', 'lamb', 'goat', 'nyama', 'fish',
        'tilapia', 'omena', 'prawn', 'shrimp', 'egg', 'lentil', 'dal', 'moong',
        'ndengu', 'kamande', 'bean', 'njahi', 'chickpea', 'tofu', 'paneer',
        'yogurt', 'curd', 'mursik', 'milk', 'cheese', 'soya', 'soybean',
        'bacon', 'ham', 'turkey', 'tuna'
    ],
    'Carbs': [
        'rice', 'ugali', 'chapati', 'mandazi', 'githeri', 'mokimo', 'matoke',
        'cassava', 'nduma', 'arrowroot', 'flour', 'atta', 'maida', 'bread',
        'bun', 'pasta', 'lasagna', 'noodle', 'potato', 'corn', 'oat', 'sugar',
        'honey', 'tortilla', 'crepe', 'dough', 'poha', 'semolina', 'rava', 'quinoa'
    ],
    'Fats': [
        'oil', 'butter', 'ghee', 'cream', 'almond', 'walnut', 'cashew',
        'peanut', 'groundnut', 'macadamia', 'nut', 'seed', 'flaxseed', 'chia',
        'simsim', 'sesame', 'coconut', 'avocado', 'mayonnaise', 'chocolate', 'olive'
    ],
    'Fiber_Vitamins': [
        'carrot', 'spinach', 'sukuma', 'managu', 'terere', 'sagaa', 'kunde',
        'tomato', 'onion', 'kachumbari', 'pepper', 'capsicum', 'broccoli',
        'cauliflower', 'pea', 'cabbage', 'kale', 'lettuce', 'cucumber', 'garlic',
        'ginger', 'cilantro', 'coriander', 'mushroom', 'mango', 'banana',
        'apple', 'orange', 'kiwi', 'lemon', 'pawpaw', 'papaya', 'pineapple',
        'vegetable', 'herb', 'basil'
    ]
}

# Practical ingredient add-ons tailored by dietary preference
VEG_ADDONS = {
    'Protein': ['lentils (ndengu/dal)', 'beans or chickpeas', 'tofu or paneer', 'Greek yogurt'],
    'Carbs': ['steamed brown rice', 'whole-wheat chapati', 'roasted sweet potatoes', 'ugali or quinoa'],
    'Fats': ['sliced avocado', 'a drizzle of olive oil', 'crushed peanuts, walnuts, or almonds', 'flaxseeds/simsim'],
    'Fiber_Vitamins': ['steamed sukuma wiki or spinach', 'fresh kachumbari (tomatoes & onions)', 'steamed cabbage or carrots', 'broccoli']
}

NON_VEG_ADDONS = {
    'Protein': ['grilled chicken breast', 'boiled eggs', 'pan-seared tilapia or prawns', 'lentils or beans'],
    'Carbs': VEG_ADDONS['Carbs'],
    'Fats': VEG_ADDONS['Fats'],
    'Fiber_Vitamins': VEG_ADDONS['Fiber_Vitamins']
}

TARGET_COLS = ['Protein', 'Carbs', 'Fats', 'Fiber_Vitamins']
THRESHOLD = 0.50


# ==========================================
# DATA PREPARATION (FOOD + RATINGS)
# ==========================================
def load_and_prepare_data(food_path='food.csv', ratings_path='ratings.csv'):
    """Loads food.csv, merges mean user ratings from ratings.csv, and tags nutrients."""
    food_df = pd.read_csv(food_path)
    ratings_df = pd.read_csv(ratings_path)

    df = food_df.copy()
    df['Describe'] = df['Describe'].fillna('').str.lower()

    # Aggregate average Rating and count per Food_ID from ratings.csv
    rating_stats = ratings_df.groupby('Food_ID').agg(
        Avg_Rating=('Rating', 'mean'),
        Rating_Count=('Rating', 'count')
    ).reset_index()

    df = pd.merge(df, rating_stats, on='Food_ID', how='left')
    global_mean = ratings_df['Rating'].mean()
    df['Avg_Rating'] = df['Avg_Rating'].fillna(global_mean).round(2)
    df['Rating_Count'] = df['Rating_Count'].fillna(0).astype(int)

    # Auto-label nutrient groups
    for nutrient, keywords in NUTRIENT_KEYWORDS.items():
        df[nutrient] = df['Describe'].apply(
            lambda text: 1 if any(word in text for word in keywords) else 0
        )
    return df


# ==========================================
# STAGE 1: NUTRIENT GAP DETECTOR (ML MODEL)
# ==========================================
def build_anchor_examples(repeat=5):
    """Creates single-ingredient anchor rows so rare ingredients aren't diluted."""
    rows = []
    for _ in range(repeat):
        for nutrient, words in NUTRIENT_KEYWORDS.items():
            for word in words:
                row = {'Describe': word}
                for col in TARGET_COLS:
                    row[col] = 1 if col == nutrient else 0
                rows.append(row)
    return pd.DataFrame(rows)


def train_nutrient_model(df):
    """Trains and evaluates the Multi-Label TF-IDF + Logistic Regression classifier."""
    X_train, X_test, y_train, y_test = train_test_split(
        df['Describe'], df[TARGET_COLS], test_size=0.2, random_state=42
    )

    anchors_df = build_anchor_examples(repeat=5)
    X_train_aug = pd.concat([X_train, anchors_df['Describe']], ignore_index=True)
    y_train_aug = pd.concat([y_train, anchors_df[TARGET_COLS]], ignore_index=True)

    vectorizer = TfidfVectorizer(stop_words='english', max_features=2000)
    X_train_vec = vectorizer.fit_transform(X_train_aug)
    X_test_vec = vectorizer.transform(X_test)

    classifier = OneVsRestClassifier(
        LogisticRegression(C=8.0, class_weight='balanced', max_iter=1000)
    )
    classifier.fit(X_train_vec, y_train_aug)

    y_test_probs = classifier.predict_proba(X_test_vec)
    y_test_pred = (y_test_probs >= THRESHOLD).astype(int)

    print("--- Stage 1 Model Evaluation Report ---")
    print(classification_report(y_test, y_test_pred, target_names=TARGET_COLS, zero_division=0))

    return vectorizer, classifier


def diagnose_meal(user_meal_text, vectorizer, classifier):
    """Predicts present and missing nutrients across the meal and its components."""
    parts = [p.strip() for p in re.split(r',|\band\b|\bwith\b|\bplus\b', user_meal_text.lower()) if p.strip()]
    texts_to_score = [user_meal_text.lower()] + parts

    vecs = vectorizer.transform(texts_to_score)
    max_probs = classifier.predict_proba(vecs).max(axis=0)

    present = [n for n, p in zip(TARGET_COLS, max_probs) if p >= THRESHOLD]
    missing = [n for n, p in zip(TARGET_COLS, max_probs) if p < THRESHOLD]

    return {
        'input_meal': user_meal_text,
        'present_nutrients': present,
        'missing_nutrients': missing,
        'is_balanced': len(missing) == 0
    }


# ==========================================
# STAGE 2: RATING-WEIGHTED MEAL RECOMMENDER
# ==========================================
def recommend_meal_improvements(diagnosis, enriched_df, is_veg=None, cuisine=None, top_n=3):
    """Recommends ingredient add-ons and top-rated complementary dishes from food.csv."""
    missing = diagnosis['missing_nutrients']

    if not missing:
        return {
            'status': 'Balanced Meal!',
            'message': 'Great job! Your meal already covers Protein, Carbs, Healthy Fats, and Fiber/Vitamins.',
            'ingredient_additions': {},
            'recommended_sides': pd.DataFrame()
        }

    # 1. Pick diet-appropriate quick ingredients to add
    addon_bank = VEG_ADDONS if (is_veg is True or str(is_veg).lower() == 'veg') else NON_VEG_ADDONS
    additions = {nutrient: addon_bank[nutrient][:3] for nutrient in missing}

    # 2. Filter dataset by dietary preference and optional cuisine
    candidates = enriched_df.copy()
    if is_veg is True or str(is_veg).lower() == 'veg':
        candidates = candidates[candidates['Veg_Non'].str.lower() == 'veg']
    elif is_veg is False or str(is_veg).lower() == 'non-veg':
        candidates = candidates[candidates['Veg_Non'].str.lower() == 'non-veg']

    if cuisine:
        cuisine_matches = candidates[candidates['C_Type'].str.lower() == cuisine.lower()]
        if not cuisine_matches.empty:
            candidates = cuisine_matches

    # 3. Compute composite Balance Score (Nutrients Covered + Healthy Bonus + User Rating + Similarity)
    candidates['Missing_Covered'] = candidates[missing].sum(axis=1)
    candidates['Healthy_Bonus'] = (candidates['C_Type'].str.lower() == 'healthy food').astype(int) * 0.4
    candidates['Rating_Boost'] = (candidates['Avg_Rating'] / 10.0) * 0.5

    target_query = " ".join([" ".join(NUTRIENT_KEYWORDS[m]) for m in missing])
    sim_vec = TfidfVectorizer(stop_words='english')
    tfidf_mat = sim_vec.fit_transform(candidates['Describe'])
    query_vec = sim_vec.transform([target_query])
    candidates['Similarity'] = cosine_similarity(query_vec, tfidf_mat).flatten()

    candidates['Balance_Score'] = (
        candidates['Missing_Covered']
        + candidates['Healthy_Bonus']
        + candidates['Rating_Boost']
        + candidates['Similarity']
    )

    valid_sides = candidates[candidates['Missing_Covered'] > 0]
    top_sides = valid_sides.sort_values(by='Balance_Score', ascending=False).head(top_n)

    display_cols = ['Food_ID', 'Name', 'C_Type', 'Veg_Non', 'Avg_Rating'] + missing
    return {
        'status': 'Needs Balancing',
        'message': f"Your meal is missing: {', '.join(missing)}.",
        'ingredient_additions': additions,
        'recommended_sides': top_sides[display_cols]
    }


# --- Execute End-to-End Test ---
if __name__ == '__main__':
    enriched_df = load_and_prepare_data('food.csv', 'ratings.csv')
    tfidf_vec, nutrient_clf = train_nutrient_model(enriched_df)

    test_meals = [
        "Ugali and tilapia",
        "Rice, Chicken, avocado, cabbage",
        "Chapati and ndengu",
        "Ugali, sukuma wiki, beef, avocado",
        "Rice and grilled chicken",
        "Plain pasta with butter"
    ]

    for meal in test_meals:
        diag = diagnose_meal(meal, tfidf_vec, nutrient_clf)
        rec = recommend_meal_improvements(diag, enriched_df, is_veg='veg')

        print("\n==========================================")
        print(f"INPUT MEAL : {diag['input_meal']}")
        print(f"PRESENT    : {diag['present_nutrients']}")
        print(f"MISSING    : {diag['missing_nutrients']}")
        print(f"VERDICT    : {rec['status']} — {rec['message']}")

        if rec['ingredient_additions']:
            print("ADD TO MEAL:", rec['ingredient_additions'])
        if not rec['recommended_sides'].empty:
            print("\nTOP RATED COMPLEMENTARY DISHES FROM DATASET:")
            print(rec['recommended_sides'].to_string(index=False))

--- Stage 1 Model Evaluation Report ---
                precision    recall  f1-score   support

       Protein       0.90      0.95      0.92        56
         Carbs       0.88      0.98      0.92        50
          Fats       0.94      0.98      0.96        62
Fiber_Vitamins       0.95      0.91      0.93        64

     micro avg       0.92      0.95      0.93       232
     macro avg       0.92      0.95      0.93       232
  weighted avg       0.92      0.95      0.93       232
   samples avg       0.90      0.93      0.90       232


INPUT MEAL : Ugali and tilapia
PRESENT    : ['Protein', 'Carbs']
MISSING    : ['Fats', 'Fiber_Vitamins']
VERDICT    : Needs Balancing — Your meal is missing: Fats, Fiber_Vitamins.
ADD TO MEAL: {'Fats': ['sliced avocado', 'a drizzle of olive oil', 'crushed peanuts, walnuts, or almonds'], 'Fiber_Vitamins': ['steamed sukuma wiki or spinach', 'fresh kachumbari (tomatoes & onions)', 'steamed cabbage or carrots']}

TOP RATED COMPLEMENTARY DISHES FROM DAT